# Assignment 1, Task A: Classification problem.

## The data:
In this QSAR exercise, the mutagenicity of various molecules is to be investigated. The dataset in use is the Ames Mutagenicity Dataset for Multi-Task learning accessed via the PyTDC library, essentially as also provided here: https://huggingface.co/datasets/scikit-fingerprints/TDC_ames. Columns have been renamed for enhanced clarity.

The dataset gives the overal mutagenicity (1 = mutagen) of various drugs (simply represented as their SMILES string). From the SMILES strings, molecular fingerprints can be generated as molecular descriptors.

## The tasks:
1) Inspect the data and clean if needed. Adhere to good practices!
2) Calculate the fingerprints (partial snippet provided) and create a feature matrix X and a target vector y
3) Then four different models should be trained on the fingerprints and evaluated according to accuracy and their roc-auc score to compare their performance. For each model, additionally, the overfitting needs to be addressed.

These four models have to be compared:
- `KNeighborsClassifier`: choose a suitable number of neighbors
- `DecisionTreeClassifier`: use a random_state
- `RandomForestClassifier`: use a random_state and a slightly bigger forest (e.g. 200 trees)
- `GradientBoostingClassifier`: use a random_state

Other than the stated parameters, the models can be mostly used as provided by `scikit`. No hyperparameter tuning needs to be performed, no CV necessary.

4) Conclusion and discussion: Provide answers to the questions.

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, ConfusionMatrixDisplay

In [3]:
df = pd.read_csv("ames_data.csv")
df


,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1
...,...,...,...
7273,Drug 7587,CCCCCCCCCCCCOCCO,0
7274,Drug 7588,CC(CCc1ccccc1)c1ccccc1,0
7275,Drug 7593,CCOP(=S)(CC)Sc1ccccc1,0
7276,Drug 7598,C=C(C)C1CC=C(C)C(OC(C)=O)C1,0


## 1. Inspect and clean the data
- Gain some overview of the data and assess NaNs and duplicates and clean if needed.
- Inspect the class balance!

In [5]:
df.info() # no dropNa needed

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7278 entries, 0 to 7277
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   drug_id       7278 non-null   object
 1   smiles        7278 non-null   object
 2   mutagenicity  7278 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 170.7+ KB


In [ ]:
df.duplicated().sum() # no dulicates -> data is clean

np.int64(0)

In [18]:
# Absolute counts
print("Class counts:")
print(df.value_counts().sum())

# Relative frequencies
print("Class proportions:")
print(df.value_counts(normalize=True).sum())

Class counts:
7278
Class proportions:
0.9999999999999999


Has 7278 entries and 7278 classes -> each one is unique 

Class proportions is preaty much 1 :)

## 2. Create fingerprints from the Smiles
The partial snippet for MorganFingerprints can be used. Note that instead of a dataframe, the function will produce a np.array, which will be written into a list. From this you can create the feature matrix and the target vector. Inspect the shape of the arrays!

In [22]:
def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fp = mfpgen.GetFingerprint(mol)
    return np.array(fp)

# Convert to fingerprints
fps = []
valid_labels = []

for smiles, label in zip(df["smiles"], df["mutagenicity"]):
    fp = smiles_to_fp(smiles)
    if fp is not None:
        fps.append(fp)
        valid_labels.append(label)


In [28]:
X = np.vstack(fps)  # shape: (n_samples, 2048)
print("Feature matrix shape:", X.shape)

Feature matrix shape: (7278, 2048)


In [33]:
y = np.array(valid_labels)
print("Target vector shape:", y.shape)

Target vector shape: (7278,)


## 3. Train the models
Use a classic train-test split of 0.2 including a random seed and `stratify`. For training and predicting labels, take note of the time the process takes for each model (does not necessarily have to be coded, can also be estimated). Make sure to predict labels for both training and test splits in order to identify overfitting. Use the accuracy and roc-auc as metrics for evaluation.

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    #random_state=42,
    stratify=y
)

In [39]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    
    # Measure training time
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Probabilities for ROC-AUC
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)

    # Print results
    print(name)
    print(f"  Train Accuracy: {train_acc:.3f}")
    print(f"  Test  Accuracy: {test_acc:.3f}")
    print(f"  Train ROC-AUC:  {train_auc:.3f}")
    print(f"  Test  ROC-AUC:  {test_auc:.3f}")
    print(f"  Training Time:  {train_time:.3f} seconds")
    print("-" * 40)



In [41]:
# KNN 
knn = KNeighborsClassifier(n_neighbors=11)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

# Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)

evaluate_model("KNN", knn, X_train, X_test, y_train, y_test)
evaluate_model("Decision Tree", dt, X_train, X_test, y_train, y_test)
evaluate_model("Random Forest", rf, X_train, X_test, y_train, y_test)
evaluate_model("Gradient Boosting", gb, X_train, X_test, y_train, y_test)

KNN
  Train Accuracy: 0.817
  Test  Accuracy: 0.786
  Train ROC-AUC:  0.903
  Test  ROC-AUC:  0.876
  Training Time:  0.004 seconds
----------------------------------------
Decision Tree
  Train Accuracy: 0.999
  Test  Accuracy: 0.787
  Train ROC-AUC:  1.000
  Test  ROC-AUC:  0.785
  Training Time:  2.103 seconds
----------------------------------------
Random Forest
  Train Accuracy: 0.999
  Test  Accuracy: 0.843
  Train ROC-AUC:  1.000
  Test  ROC-AUC:  0.922
  Training Time:  2.615 seconds
----------------------------------------
Gradient Boosting
  Train Accuracy: 0.810
  Test  Accuracy: 0.797
  Train ROC-AUC:  0.891
  Test  ROC-AUC:  0.878
  Training Time:  25.551 seconds
----------------------------------------


## 4. Conclusion and discussion
- Which model performed the best?
- Which was the most time efficient?
- Which model showed the wors overfitting?
- Why does ensemble learning outperform a single tree?
- Why does KNN perform well in high-dimensional fingerprint space?
- What does ROC-AUC tell us that accuracy does not?

Which model performed the best? 
- Random Forest (highest Test Accuracy = 0.843 and highest Test ROC-AUC = 0.922).

Which was the most time efficient? 
-  KNN (0.004 s training time).

Which model showed the worst overfitting? 
-  Decision Tree \
    - Train Accuracy = 0.999 vs Test Accuracy = 0.787 \
    - Train ROC-AUC = 1.000 vs Test ROC-AUC = 0.785 
-  high variance aka biggets difference in between Testing and Training

Why does ensemble learning outperform a single tree? 
- Ensembles (Random Forest) reduce variance by aggregating many decorrelated trees. 
- A single tree has high variance and easily overfits; averaging trees stabilizes predictions and improves generalization.

Why does KNN perform well in high-dimensional fingerprint space? 
- Fingerprints encode structural similarity directly (meaningful similarity space) -> structurally similar molecules share many fingerprint parts 
- Chemical activity often follows the similarity principle -> KNN exploits this via local neighborhood decisions 
- Feature vectors are high-dimensional but sparse, so distances are still informative 

What does ROC-AUC tell us that accuracy does not? 
- Measures ranking quality of predictions, not just final class labels 
- Evaluates performance across all classification thresholds 
- Is threshold-independent (accuracy depends on a fixed cutoff, usually 0.5) 
- Is more robust to class imbalance #
